In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd

# Data Loading 

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv',
    encoding='latin-1')
df.sample(5)
df.shape

# data cleaning

In [ ]:
df.info()

In [ ]:
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)

In [ ]:
df.sample(5)

In [ ]:
#rename 
df.rename(columns={'v1':'target','v2':'text'},inplace=True)
df.sample(5)

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])
print(df['target'])
df.head()

In [ ]:
# missing values
print("missing values ",df.isnull().sum())
# check for duplicate values
print("duplicate values ",df.duplicated(subset='text').sum())
# remove duplicates (we dont have duplicate values )
df = df.drop_duplicates(keep='first')
print("df",df)
print("after oprtn ",df.duplicated().sum())
df.shape

# EDA

In [ ]:
print(df.head())
print(df['target'].value_counts())
import matplotlib.pyplot as plt
plt.pie(df['target'].value_counts(), labels=['ham','spam'],autopct="%0.2f")
plt.show()


In [ ]:
import nltk
import seaborn as sns
# ==============================
# Feature Engineering
# ==============================

df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(lambda x: len(nltk.word_tokenize(x)))
df['num_sentences'] = df['text'].apply(lambda x: len(nltk.sent_tokenize(x)))

# ==============================
# Show Sample Data
# ==============================

print("\n🔹 Sample Data:")
print(df.head())

# ==============================
# Overall Statistics
# ==============================

print("\n Overall Statistics:")
print(df[['num_characters','num_words','num_sentences']].describe().round(2))

# ==============================
# Ham vs Spam Comparison
# ==============================

print("\n HAM Messages (0):")
print(df[df['target'] == 0][['num_characters','num_words','num_sentences']].describe().round(2))

print("\nSPAM Messages (1):")
print(df[df['target'] == 1][['num_characters','num_words','num_sentences']].describe().round(2))

# ==============================
# Visualizations
# ==============================

# Characters Distribution
plt.figure(figsize=(10,5))
sns.histplot(df[df['target'] == 0]['num_characters'], label='Ham')
sns.histplot(df[df['target'] == 1]['num_characters'], label='Spam')
plt.legend()
plt.title("Character Count Distribution")
plt.show()

# Words Distribution
plt.figure(figsize=(10,5))
sns.histplot(df[df['target'] == 0]['num_words'], label='Ham')
sns.histplot(df[df['target'] == 1]['num_words'], label='Spam')
plt.legend()
plt.title("Word Count Distribution")
plt.show()

# Pairplot (relationships)
sns.pairplot(df[['num_characters','num_words','num_sentences','target']], hue='target')
plt.show()

# Heatmap
plt.figure(figsize=(8,5))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True)
plt.title("Correlation Heatmap")
plt.show()

# Data preprocessing


* Lower case
* Tokenization
* Removing special characters
* Removing stop words and punctuation
* Stemming


In [ ]:

import string
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from wordcloud import WordCloud
from collections import Counter

# download once
nltk.download('punkt')
nltk.download('stopwords')

# initialize stemmer
ps = PorterStemmer()

# ==============================
# Text Preprocessing Function
# ==============================

def transform_text(text):
    # lowercase
    text = text.lower()
    
    # tokenize
    words = nltk.word_tokenize(text)
    
    # remove non-alphanumeric
    words = [word for word in words if word.isalnum()]
    
    # remove stopwords & punctuation
    words = [word for word in words if word not in stopwords.words('english')]
    
    # stemming
    words = [ps.stem(word) for word in words]
    
    return " ".join(words)

# ==============================
# Apply Transformation
# ==============================

df['transformed_text'] = df['text'].apply(transform_text)

print("\n Transformed Data Sample:")
print(df[['text','transformed_text']].head())

# ==============================
# WordClouds
# ==============================

wc = WordCloud(width=500, height=500, min_font_size=10, background_color='white')

# Spam WordCloud
spam_text = df[df['target'] == 1]['transformed_text'].str.cat(sep=" ")
spam_wc = wc.generate(spam_text)

plt.figure(figsize=(10,5))
plt.imshow(spam_wc)
plt.title("Spam WordCloud")
plt.axis('off')
plt.show()

# Ham WordCloud
ham_text = df[df['target'] == 0]['transformed_text'].str.cat(sep=" ")
ham_wc = wc.generate(ham_text)

plt.figure(figsize=(10,5))
plt.imshow(ham_wc)
plt.title("Ham WordCloud")
plt.axis('off')
plt.show()

# ==============================
#  Most Common Words (Spam)
# ==============================

spam_words = []
for msg in df[df['target'] == 1]['transformed_text']:
    spam_words.extend(msg.split())

spam_common = Counter(spam_words).most_common(30)
spam_df = pd.DataFrame(spam_common, columns=['word','count'])

plt.figure(figsize=(10,5))
sns.barplot(data=spam_df, x='word', y='count')
plt.xticks(rotation='vertical')
plt.title("Top 30 Spam Words")
plt.show()

# ==============================
# Most Common Words (Ham)
# ==============================

ham_words = []
for msg in df[df['target'] == 0]['transformed_text']:
    ham_words.extend(msg.split())

ham_common = Counter(ham_words).most_common(30)
ham_df = pd.DataFrame(ham_common, columns=['word','count'])

plt.figure(figsize=(10,5))
sns.barplot(data=ham_df, x='word', y='count')
plt.xticks(rotation='vertical')
plt.title("Top 30 Ham Words")
plt.show()

# Model Building

# Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

print("Shape of X:", X.shape)

# Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# Naive Bayes Comparison

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix

models_nb = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB()
}

for name, model in models_nb.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Multiple Model Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier
)
from xgboost import XGBClassifier

models = {
    "NB": MultinomialNB(),
    "LR": LogisticRegression(solver='liblinear'),
    "SVC": SVC(kernel='sigmoid', gamma=1.0),
    "DT": DecisionTreeClassifier(max_depth=5),
    "KNN": KNeighborsClassifier(),
    "RF": RandomForestClassifier(n_estimators=50, random_state=2),
    "AdaBoost": AdaBoostClassifier(n_estimators=50, random_state=2),
    "Bagging": BaggingClassifier(n_estimators=50, random_state=2),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=50, random_state=2),
    "GBDT": GradientBoostingClassifier(n_estimators=50, random_state=2),
    "XGB": XGBClassifier(n_estimators=50, random_state=2)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    
    print(f"\n{name}")
    print("Accuracy:", acc)
    print("Precision:", prec)
    
    results.append([name, acc, prec])

# Results DataFrame

In [ ]:
import pandas as pd

performance_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision'])
performance_df = performance_df.sort_values('Precision', ascending=False)

print(performance_df)

# Visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.catplot(
    data=pd.melt(performance_df, id_vars="Model"),
    x="Model", y="value", hue="variable",
    kind="bar", height=5
)

plt.ylim(0.5, 1.0)
plt.xticks(rotation=45)
plt.show()

# Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier

svc = SVC(kernel='sigmoid', gamma=1.0, probability=True)
mnb = MultinomialNB()
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)

voting = VotingClassifier(
    estimators=[('svm', svc), ('nb', mnb), ('et', etc)],
    voting='soft'
)

voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)

print("Voting Accuracy:", accuracy_score(y_test, y_pred))
print("Voting Precision:", precision_score(y_test, y_pred))

# Stacking Classifier

In [ ]:
from sklearn.ensemble import StackingClassifier

estimators = [('svm', svc), ('nb', mnb), ('et', etc)]
final_estimator = RandomForestClassifier()

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator
)

stacking.fit(X_train, y_train)
y_pred = stacking.predict(X_test)

print("Stacking Accuracy:", accuracy_score(y_test, y_pred))
print("Stacking Precision:", precision_score(y_test, y_pred))

# Save Model

In [ ]:
import pickle
from sklearn.naive_bayes import MultinomialNB

# Train model 
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

# ✅ Save trained objects
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('model.pkl', 'wb'))

print("✅ Model trained and saved correctly!")